<a href="https://colab.research.google.com/github/pyrenaaaaaa/Emotion-Aware-Companion/blob/text/FINAL_UPDATED_(Split_Train_Test)_and_FINETUNED_Working_Copy_of_Llama_3_2_3B_(11_03_2025).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# # Mount Google Drive
from google.colab import drive
drive. mount ('/content/drive')

Mounted at /content/drive


In [2]:
from IPython.display import display, Javascript

def prevent_disconnect():
    display(Javascript('''
        function keepColabAlive() {
            setInterval(() => {
                console.log("Preventing Colab disconnect...");
                document.querySelector("colab-toolbar-button#connect").click();
            }, 60000);  // Clicks "Reconnect" every 60 seconds
        }
        keepColabAlive();
    '''))

prevent_disconnect()

<IPython.core.display.Javascript object>

In [3]:
# Install Unsloth and dependencies
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Install only in Colab to avoid conflicts
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

    !pip install wandb --upgrade


# os.environ["WANDB_DISABLED"] = "true"  # Disable WandB logging

# Restart kernel after installing
import IPython
IPython.display.clear_output()

# Imports


In [4]:
# Ensure Unsloth is imported FIRST before Transformers, TRL, PEFT
import unsloth
from unsloth import FastLanguageModel

import torch
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset, DatasetDict
from transformers import BitsAndBytesConfig, TextStreamer
from accelerate import infer_auto_device_map
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from unsloth.chat_templates import standardize_sharegpt
from unsloth.chat_templates import get_chat_template
from datasets import Dataset, concatenate_datasets

import glob
import os
import shutil
import matplotlib.pyplot as plt
from unsloth.chat_templates import train_on_responses_only

from sentence_transformers import SentenceTransformer, util
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import AutoModelForCausalLM, AutoTokenizer

from peft import LoraConfig, PeftModel
import wandb

# Check GPU info
gpu_info = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_info.name}, Total Memory: {gpu_info.total_memory / 1e9:.2f} GB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.5.1+cu121 with CUDA 1201 (you have 2.6.0+cu124)
    Python  3.11.11 (you have 3.11.11)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA A100-SXM4-40GB, Total Memory: 42.47 GB


# Load & Merge Conversational Datasets

# Loading, Merging & Preprocessing Reddit Mental Health Data (Current)

In [5]:
# Define dataset path
DATA_PATH = "/content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data"

# Load raw Reddit data
raw_data_path = os.path.join(DATA_PATH, "raw data")
all_csv_files = glob.glob(os.path.join(raw_data_path, "*/*/*.csv"), recursive=True)

# Load data in chunks to prevent memory overload
data_chunks = []
for file in all_csv_files:
    print(f"Loading {file}...")
    chunk = pd.read_csv(file, usecols=["title", "selftext", "subreddit", "timestamp"], low_memory=False)
    chunk.dropna(subset=["selftext"], inplace=True)
    data_chunks.append(chunk)

raw_reddit_data = pd.concat(data_chunks, ignore_index=True)
raw_reddit_data.columns = raw_reddit_data.columns.str.strip().str.lower()
raw_reddit_data["timestamp"] = pd.to_datetime(raw_reddit_data["timestamp"], errors="coerce")

# ✅ CHANGE: Filter Out Low-Quality Data & Limit by Date
banned_words = ["[deleted]", "[removed]", "spam", "troll", "bot"]
filtered_reddit_data = raw_reddit_data[~raw_reddit_data["selftext"].str.lower().isin(banned_words)]
filtered_reddit_data = filtered_reddit_data[filtered_reddit_data["timestamp"] >= "2021-01-01"]

# ✅ CHANGE: Downsample to 500,000 records
max_samples = 500000
if len(filtered_reddit_data) > max_samples:
    filtered_reddit_data = filtered_reddit_data.sample(n=max_samples, random_state=42)

filtered_reddit_data = filtered_reddit_data[["title", "selftext", "subreddit", "timestamp"]]

print("Filtered Reddit dataset size:", filtered_reddit_data.shape)

# Load labeled Reddit data
labeled_data_path = os.path.join(DATA_PATH, "Labelled Data")
label_files = glob.glob(os.path.join(labeled_data_path, "*.csv"))
labeled_data_list = [pd.read_csv(file) for file in label_files]
labeled_reddit_data = pd.concat(labeled_data_list, ignore_index=True)
labeled_reddit_data.columns = labeled_reddit_data.columns.str.strip().str.lower()
labeled_reddit_data = labeled_reddit_data[["title", "selftext", "subreddit", "label"]].dropna(subset=["selftext"])


Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Feb21/anxifeb21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Feb21/depfeb21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Feb21/lonefeb21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Feb21/mhfeb21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Feb21/swfeb21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Jul 21/depjul21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/2021/Jul 21/anxijul21.csv...
Loading /content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/raw data/

# Format Data into LlaMA Chat Format (Current)

In [6]:
# def format_conversations(df, labeled=False):
#     """
#     Convert dataset into LLaMA's structured chat format.
#     """
#     conversations = []
#     for _, row in df.iterrows():
#         user_text = f"<|start_header_id|>user<|end_header_id|>\n\n{row['title']}\n{row['selftext']}\n<|eot_id|>\n"
#         assistant_text = f"<|start_header_id|>assistant<|end_header_id|>\n\n"
#         if labeled:
#             assistant_text += f"Category: {row['label']}"
#         conversations.append(f"{user_text}{assistant_text}<|eot_id|>")
#     return conversations

# raw_reddit_data["conversations"] = format_conversations(raw_reddit_data)
# labeled_reddit_data["conversations"] = format_conversations(labeled_reddit_data, labeled=True)

def format_conversations(df):
    conversations = []
    for _, row in df.iterrows():
        user_text = f"<|start_header_id|>user<|end_header_id|>\n\n{row['title']}\n{row['selftext']}\n<|eot_id|>\n"
        assistant_text = f"<|start_header_id|>assistant<|end_header_id|>\n\n<|eot_id|>"
        conversations.append(f"{user_text}{assistant_text}")
    return conversations

filtered_reddit_data["text"] = format_conversations(filtered_reddit_data)

# Load other Datasets (Current)

In [7]:
# Load CBT datasets
cbt_qa = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="qa_test.json", split="train")
cbt_cd = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="distortions_test.json", split="train")
cbt_pc = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="core_major_test.json", split="train")
cbt_fc = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="core_fine_test.json", split="train")

# Load other Hugging Face datasets
fine_tome = load_dataset("mlabonne/FineTome-100k", split="train", trust_remote_code=True)
daily_dialog = load_dataset("daily_dialog", split="train", trust_remote_code=True)
empathetic_dialogues = load_dataset("facebook/empathetic_dialogues", split="train", trust_remote_code=True)

README.md:   0%|          | 0.00/5.90k [00:00<?, ?B/s]

qa_test.json:   0%|          | 0.00/103k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

distortions_test.json:   0%|          | 0.00/225k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

README.md:   0%|          | 0.00/7.15k [00:00<?, ?B/s]

empathetic_dialogues.py:   0%|          | 0.00/4.51k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76673 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10943 [00:00<?, ? examples/s]

# Check Column Structure before processing

In [ ]:
print(f"CBT QA Columns: {cbt_qa.column_names}")
print(f"CBT CD Columns: {cbt_cd.column_names}")
print(f"CBT PC Columns: {cbt_pc.column_names}")
print(f"CBT FC Columns: {cbt_fc.column_names}")
print(f"FineTome Columns: {fine_tome.column_names}")
print(f"DailyDialog Columns: {daily_dialog.column_names}")
print(f"EmpatheticDialogues Columns: {empathetic_dialogues.column_names}")

CBT QA Columns: ['id', 'question', 'a', 'b', 'c', 'd', 'e']
CBT CD Columns: ['id', 'ori_text', 'situation', 'thoughts', 'distortions']
CBT PC Columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_major']
CBT FC Columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_fine_grained']
FineTome Columns: ['conversations', 'source', 'score']
DailyDialog Columns: ['dialog', 'act', 'emotion']
EmpatheticDialogues Columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags']


# Format & Remove Extra Columns

In [10]:
import re
# ✅ Format CBT QA Without Removing Non-Existent Columns (Fixed)
def format_cbt_qa(example):
    options = f"A) {example.get('a', '')} B) {example.get('b', '')} C) {example.get('c', '')} D) {example.get('d', '')}"
    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{example['question']}\n{options}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nCorrect Answer: {example.get('d', 'Unknown')}\n<|eot_id|>"}

cbt_qa = cbt_qa.map(format_cbt_qa)  # ✅ Removed `remove_columns` to fix error

# ✅ Format CBT CD (Fixed)
def format_cbt_cd(example):
    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{example['situation']}\n"
                    f"Thoughts: {example['thoughts']}\nDistortions: {example['distortions']}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nCognitive Restructuring Advice:\n<|eot_id|>"}

cbt_cd = cbt_cd.map(format_cbt_cd)  # ✅ No need to remove columns

# ✅ Format CBT PC (Fixed)
def format_cbt_pc(example):
    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{example['situation']}\n"
                    f"Thoughts: {example['thoughts']}\nCore Belief Major: {example['core_belief_major']}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nTherapeutic Response:\n<|eot_id|>"}

cbt_pc = cbt_pc.map(format_cbt_pc)  # ✅ No need to remove columns

# ✅ Format CBT FC (Fixed)
def format_cbt_fc(example):
    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{example['situation']}\n"
                    f"Thoughts: {example['thoughts']}\nCore Belief Fine Grained: {example['core_belief_fine_grained']}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nTherapeutic Response:\n<|eot_id|>"}

cbt_fc = cbt_fc.map(format_cbt_fc)  # ✅ No need to remove columns

# ✅ Fix FineTome Formatting (Handles JSON-Like Conversations)
def format_finetome(example):
    if isinstance(example['conversations'], list):
        conversation_text = "\n".join([f"{turn['from']}: {turn['value']}" for turn in example['conversations']])
    else:
        conversation_text = example['conversations']
    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{conversation_text}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nResponse:\n<|eot_id|>"}

fine_tome = fine_tome.map(format_finetome)  # ✅ No need to remove columns

# ✅ Fix Empathetic Dialogues Formatting (Removes `_comma_` Artifacts)
def clean_text(text):
    text = re.sub(r'_(comma_|period_)', '', text)  # Remove `_comma_`, `_period_`
    text = text.replace(" ,", ",").replace(" .", ".")  # Fix spaces before punctuation
    return text.strip()

def format_empathetic_dialogues(example):
    cleaned_context = clean_text(example['context'])
    cleaned_utterance = clean_text(example['utterance'])
    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{cleaned_context} {cleaned_utterance}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nEmpathetic Response:\n<|eot_id|>"}

empathetic_dialogues = empathetic_dialogues.map(format_empathetic_dialogues)

def format_dailydialog(example):
    """
    Formats DailyDialog dataset into structured conversation.
    """
    if isinstance(example["dialog"], list):
        conversation_text = "\n".join(example["dialog"])  # Convert list to structured dialogue
    else:
        conversation_text = str(example["dialog"])  # Ensure it's always a string

    return {"text": f"<|start_header_id|>user<|end_header_id|>\n\n{conversation_text}\n\n"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\nResponse:\n<|eot_id|>"}

# ✅ Apply Fix: Properly Convert DailyDialog to Single "text" Field
daily_dialog = daily_dialog.map(format_dailydialog).remove_columns(['dialog', 'act', 'emotion'])

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

Map:   0%|          | 0/184 [00:00<?, ? examples/s]

Map:   0%|          | 0/112 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/76673 [00:00<?, ? examples/s]

Map:   0%|          | 0/11118 [00:00<?, ? examples/s]

# Ensure 'text' Column exists & convert to string (Current)

In [11]:
def create_text_column(example):
    """
    Ensures every dataset entry has a valid 'text' column and converts everything to a string.
    """
    if example.get("conversations"):
        if isinstance(example["conversations"], list):  # If it's a list, join into a string
            conversation_text = "\n".join([f"{turn['from']}: {turn['value']}" for turn in example["conversations"]])
        else:
            conversation_text = str(example["conversations"])
        return {"text": conversation_text}

    elif example.get("title") and example.get("selftext"):
        return {"text": f"user:\n{example['title']}\n{example['selftext']}\n\nassistant:\n"}

    elif example.get("selftext"):
        return {"text": f"user:\n{example['selftext']}\n\nassistant:\n"}

    elif example.get("question"):
        options = f"A) {example.get('a', '')} B) {example.get('b', '')} C) {example.get('c', '')} D) {example.get('d', '')}"
        return {"text": f"user:\n{example['question']}\n{options}\n\nassistant:\n{example.get('d', 'Unknown Answer')}"}

    return {"text": ""}

# Standardize Dataset Columns

In [12]:
from datasets import Features, Value

# Standardization (Ensures only 'text' column exists)
common_features = Features({"text": Value("string")})

def standardize_dataset(ds):
    """
    Ensures the dataset has only the 'text' column and is correctly formatted.
    """
    return ds.remove_columns([col for col in ds.column_names if col != "text"]).cast(common_features)

datasets_to_merge = [cbt_qa, cbt_cd, cbt_pc, cbt_fc, fine_tome, daily_dialog, empathetic_dialogues]
merged_dataset = concatenate_datasets(datasets_to_merge)
merged_dataset = merged_dataset.shuffle(seed=42).select(range(min(500000, len(merged_dataset))))
merged_dataset = merged_dataset.filter(lambda ex: ex["text"] is not None and ex["text"].strip() != "")

# Verify Final Dataset
print("Final Merged Dataset Columns:", merged_dataset.column_names)
print(f"Final Merged Dataset Size: {len(merged_dataset)}")
print(f"Sample record:\n{merged_dataset[0]}")

Filter:   0%|          | 0/188453 [00:00<?, ? examples/s]

Final Merged Dataset Columns: ['id', 'question', 'a', 'b', 'c', 'd', 'e', 'text', 'ori_text', 'situation', 'thoughts', 'distortions', 'core_belief_major', 'core_belief_fine_grained', 'conversations', 'source', 'score', 'conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags']
Final Merged Dataset Size: 188453
Sample record:
{'id': None, 'question': None, 'a': None, 'b': None, 'c': None, 'd': None, 'e': None, 'text': "<|start_header_id|>user<|end_header_id|>\n\nhopeful She's in Ghana.\n\n<|start_header_id|>assistant<|end_header_id|>\n\nEmpathetic Response:\n<|eot_id|>", 'ori_text': None, 'situation': None, 'thoughts': None, 'distortions': None, 'core_belief_major': None, 'core_belief_fine_grained': None, 'conversations': None, 'source': None, 'score': None, 'conv_id': 'hit:1220_conv:2441', 'utterance_idx': 3, 'context': 'hopeful', 'prompt': "My online friend appiled to come to America. We're waiting for some good news.", 'speaker_idx': 1, 'utteranc

# Check if Datasets Works

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Convert Merged Dataset to Pandas
merged_df = merged_dataset.to_pandas()

# Perform Train-Test Split (70% Train, 30% Validation)
train_df, valid_df = train_test_split(merged_df, test_size=0.3, random_state=42)

# Convert Back to Hugging Face Dataset Format
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

# Print dataset sizes again
print(f"Total dataset size: {len(merged_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(valid_dataset)}")

# Visualize sample data
import random
sample_texts = random.sample(merged_dataset["text"], 5)
for i, text in enumerate(sample_texts):
    print(f"Sample {i+1}: {text[:500]}")  # Print first 500 characters

Total dataset size: 188453
Train dataset size: 131917
Validation dataset size: 56536
Sample 1: <|start_header_id|>user<|end_header_id|>

nostalgic Why is that

<|start_header_id|>assistant<|end_header_id|>

Empathetic Response:
<|eot_id|>
Sample 2: <|start_header_id|>user<|end_header_id|>

I am really thirsty . 
 How about we go and get something to drink ? 
 Let's do that . 
 Do you know what you want to get ? 
 A soda sounds good . 
 Soda isn't the best thing to drink when you're thirsty . 
 Why is that ? 
 Soda isn't good for you . 
 What should I drink then ? 
 You should really drink water . 
 That sounds good . 
 It's a lot better than soda . 

<|start_header_id|>assistant<|end_header_id|>

Response:
<|eot_id|>
Sample 3: <|start_header_id|>user<|end_header_id|>

human: How do I calculate the angle of reflection of a ball colliding with an angled surface given the ball's current position, velocity, and the start and end coordinates of the line of collision?
gpt: To calculate the a

In [ ]:
# Print dataset sizes
print(f"CBT QA Dataset size: {len(cbt_qa)}")
print(f"CBT CD Dataset size: {len(cbt_cd)}")
print(f"CBT PC Dataset size: {len(cbt_pc)}")
print(f"CBT FC Dataset size: {len(cbt_fc)}")
print(f"FineTome Dataset size: {len(fine_tome)}")
print(f"DailyDialog Dataset size: {len(daily_dialog)}")
print(f"EmpatheticDialogues Dataset size: {len(empathetic_dialogues)}")

CBT QA Dataset size: 220
CBT CD Dataset size: 146
CBT PC Dataset size: 184
CBT FC Dataset size: 112
FineTome Dataset size: 100000
DailyDialog Dataset size: 11118
EmpatheticDialogues Dataset size: 76673


In [ ]:
# ✅ Check Dataset Columns Before Printing Samples
print("\n✅ Dataset Column Names:")
print(f"CBT QA Columns: {cbt_qa.column_names}")
print(f"CBT CD Columns: {cbt_cd.column_names}")
print(f"CBT PC Columns: {cbt_pc.column_names}")
print(f"CBT FC Columns: {cbt_fc.column_names}")
print(f"FineTome Columns: {fine_tome.column_names}")
print(f"DailyDialog Columns: {daily_dialog.column_names}")
print(f"EmpatheticDialogues Columns: {empathetic_dialogues.column_names}")


✅ Dataset Column Names:
CBT QA Columns: ['id', 'question', 'a', 'b', 'c', 'd', 'e', 'text']
CBT CD Columns: ['id', 'ori_text', 'situation', 'thoughts', 'distortions', 'text']
CBT PC Columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_major', 'text']
CBT FC Columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_fine_grained', 'text']
FineTome Columns: ['conversations', 'source', 'score', 'text']
DailyDialog Columns: ['text']
EmpatheticDialogues Columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags', 'text']


In [ ]:
import random

# ✅ Function to print random sample correctly
def print_random_sample(dataset, dataset_name):
    sample = dataset[random.randint(0, len(dataset) - 1)]  # Get random sample
    # ✅ Check if 'text' column exists before accessing
    if "text" in sample:
        print(f"\n🔹 {dataset_name} Sample:\n{sample['text'][:500]}\n{'-'*80}")
    else:
        print(f"\n⚠️ {dataset_name} does not contain 'text' column. Available columns: {dataset.column_names}")

# ✅ Test CBT Datasets
print_random_sample(cbt_qa, "CBT QA")
print_random_sample(cbt_cd, "CBT CD")
print_random_sample(cbt_pc, "CBT PC")
print_random_sample(cbt_fc, "CBT FC")

# ✅ Test FineTome
print_random_sample(fine_tome, "FineTome")

# ✅ Test DailyDialog
print_random_sample(daily_dialog, "DailyDialog")

# ✅ Test EmpatheticDialogues
print_random_sample(empathetic_dialogues, "Empathetic Dialogues")


🔹 CBT QA Sample:
<|start_header_id|>user<|end_header_id|>

What does the term "reframing" refer to?
A) Encouraging clients to suppress their emotions to avoid discomfort. B) Identifying and challenging irrational thought patterns contributing to distress. C) Shifting perspectives or interpretations of situations to promote more adaptive responses. D) Utilizing mindfulness techniques to increase awareness of present-moment experiences.

<|start_header_id|>assistant<|end_header_id|>

Correct Answer: Utilizing mind
--------------------------------------------------------------------------------

🔹 CBT CD Sample:
<|start_header_id|>user<|end_header_id|>

Needless to say, marriage has been difficult for us and I am now considering leaving after 16 months. Because I come from a culture in which marriage and community are important, I am torn between that part of myself and the part that is immersed in a greater individualistic culture that values personal happiness and fulfillment.
Thoughts

# Train the Model

# Model & Tokenizer Loading

In [13]:
# import os
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from accelerate import infer_auto_device_map
# from peft import LoraConfig, get_peft_model

# # ✅ Optimize PyTorch Memory Allocation
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# torch.cuda.empty_cache()  # ✅ Free Up GPU Memory

# # ✅ Load Base Model & Tokenizer
# model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

# # ✅ Configure 4-bit Quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# # ✅ Step 1: Load Base Model (CPU First)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="cpu",
#     low_cpu_mem_usage=True,
# )

# # ✅ Step 2: Assign Model to GPU (If Available)
# device_map = infer_auto_device_map(
#     model,
#     max_memory={0: "10GB", "cpu": "6GB"}  # ✅ Adjust based on your system
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map=device_map,
#     low_cpu_mem_usage=True,
# )

# # ✅ Step 3: Apply LoRA for Fine-Tuning
# lora_config = LoraConfig(
#     r=16,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#     lora_alpha=16,
#     lora_dropout=0.05,
#     bias="none"
# )

# model = get_peft_model(model, lora_config)  # ✅ Correctly apply LoRA

# print("✅ Model loaded with LoRA configuration and ready for fine-tuning!")

# Train-Test Split (70/30)

In [14]:
from sklearn.model_selection import train_test_split

# Convert Dataset to Pandas for Splitting
merged_df = merged_dataset.to_pandas()

# Reduce dataset size to 10,000 total
reduced_df = merged_df.sample(n=10_000, random_state=42)

# Perform 70-30 Train-Test Split
train_df, valid_df = train_test_split(reduced_df, test_size=0.3, random_state=42)

# Convert Back to Hugging Face Dataset Format
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

# Print final dataset sizes
print(f"Train Dataset: {len(train_dataset)} samples")  # 7,000
print(f"Validation Dataset: {len(valid_dataset)} samples")  # 3,000

Train Dataset: 7000 samples
Validation Dataset: 3000 samples


# Training & Experiment Tracking

In [15]:
os.environ["WANDB_API_KEY"] = "6e78380a5c40f1a6a07fe34f877ad8d790bbe0a9"
wandb.login(key=os.getenv("WANDB_API_KEY"))
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pyrena-chua (pyrena-chua-singapore-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Delete old checkpont before training

In [26]:
# checkpoint_dir = "/content/drive/MyDrive/EmotionCompanion/checkpoints"
# if os.path.exists(checkpoint_dir):
#     shutil.rmtree(checkpoint_dir)  # Delete old checkpoints
#     print(f"Deleted old checkpoint directory: {checkpoint_dir}")
# else:
#     print("No previous checkpoint found, starting fresh training.")

In [ ]:
# import os

# # Define checkpoint directory
# checkpoint_dir = "/content/drive/MyDrive/EmotionCompanion/checkpoints"

# # Check if there are previous checkpoints
# last_checkpoint = None
# if os.path.exists(checkpoint_dir):
#     checkpoints = sorted([ckpt for ckpt in os.listdir(checkpoint_dir) if "checkpoint" in ckpt])
#     if checkpoints:
#         last_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])  # Load latest checkpoint
#         print(f"✅ Resuming training from {last_checkpoint}")

# # ✅ Free up GPU memory before training
# torch.cuda.empty_cache()

# # ✅ Fix 1: Ensure `use_cache=False` to prevent memory issues
# model.config.use_cache = False

# # ✅ Fix 2: Ensure Model Parameters Require Gradients
# for param in model.parameters():
#     if param.dtype in [torch.float32, torch.float16, torch.bfloat16]:  # ✅ Prevents RuntimeError
#         param.requires_grad = True

# # training_args = TrainingArguments(
# #     per_device_train_batch_size=4,
# #     gradient_accumulation_steps=8,
# #     warmup_steps=5,
# #     max_steps=500,  # Adjust if needed
# #     learning_rate=2e-4,
# #     fp16=False,  # ✅ Keep your mixed precision settings
# #     bf16=False,  # ✅ Keep your mixed precision settings
# #     eval_strategy="steps",  # ✅ Changed from evaluation_strategy (future-proof)
# #     eval_steps=250,  # Evaluate every 250 steps
# #     save_strategy="steps",
# #     save_steps=250,  # ✅ Saves checkpoint every 250 steps
# #     save_total_limit=3,  # ✅ Keeps last 3 checkpoints
# #     metric_for_best_model="loss",  # ✅ Fix: Use "loss" instead of "eval_loss"
# #     greater_is_better=False,  # ✅ Lower loss is better
# #     gradient_checkpointing=True,
# #     optim="adamw_8bit",
# #     weight_decay=0.01,
# #     lr_scheduler_type="linear",
# #     seed=3407,
# #     output_dir=checkpoint_dir,
# #     report_to="wandb",
# #     load_best_model_at_end=True,
# # )

# # def get_safe_batch_size():
# #     free_mem = torch.cuda.mem_get_info()[0] / 1e9
# #     if free_mem > 20:
# #         return 4  # High memory → batch size 4
# #     elif free_mem > 10:
# #         return 2  # Medium memory → batch size 2
# #     else:
# #         return 1  # Low memory → batch size 1 (avoids crashes)

# # training_args = TrainingArguments(
# #     per_device_train_batch_size=get_safe_batch_size(),
# #     gradient_accumulation_steps=8,
# #     warmup_steps=5,
# #     max_steps=100,  # Adjust if needed
# #     learning_rate=2e-4,
# #     fp16=False,  # ✅ Keep your mixed precision settings
# #     bf16=False,  # ✅ Keep your mixed precision settings
# #     eval_strategy="steps",  # ✅ Changed from evaluation_strategy (future-proof)
# #     eval_steps=25,  # Evaluate every 250 steps
# #     save_strategy="steps",
# #     save_steps=25,  # ✅ Saves checkpoint every 250 steps
# #     save_total_limit=1,  # ✅ Keeps last 3 checkpoints
# #     metric_for_best_model="loss",  # ✅ Fix: Use "loss" instead of "eval_loss"
# #     greater_is_better=False,  # ✅ Lower loss is better
# #     gradient_checkpointing=True,
# #     optim="adamw_8bit",
# #     weight_decay=0.01,
# #     lr_scheduler_type="linear",
# #     seed=3407,
# #     output_dir=checkpoint_dir,
# #     report_to="wandb",
# #     load_best_model_at_end=True,
# # )



# # print("Training arguments configured with checkpoint saving.")

# def get_safe_batch_size():
#     free_mem = torch.cuda.mem_get_info()[0] / 1e9
#     if free_mem > 20:
#         return 4  # High memory → batch size 4
#     elif free_mem > 10:
#         return 2  # Medium memory → batch size 2
#     else:
#         return 1  # Low memory → batch size 1 (avoids crashes)

# training_args = TrainingArguments(
#     per_device_train_batch_size=get_safe_batch_size(),
#     gradient_accumulation_steps=4,  # ✅ Reduced accumulation for better stability
#     warmup_steps=3,  # ✅ Shorter warmup for fewer training steps
#     max_steps=100,  # ✅ Reduced to prevent crashes
#     learning_rate=2e-4,
#     fp16=True,  # ✅ Enabled mixed precision to save memory
#     bf16=False,
#     eval_strategy="steps",
#     eval_steps=25,  # ✅ More frequent evaluation
#     save_strategy="steps",
#     save_steps=25,  # ✅ More frequent checkpoints
#     save_total_limit=1,  # ✅ Store only the latest checkpoint
#     metric_for_best_model="loss",
#     greater_is_better=False,
#     gradient_checkpointing=True,
#     optim="adamw_8bit",
#     weight_decay=0.01,
#     lr_scheduler_type="linear",
#     seed=3407,
#     output_dir=checkpoint_dir,
#     report_to="wandb",
#     load_best_model_at_end=True,
# )

# print("✅ Training arguments configured for stable, crash-free training.")



# Use 4-bit Quantization and LoRA for Memory Efficiency

In [ ]:
# from peft import get_peft_model, LoraConfig, PeftModel

# # Use 4-bit Quantization & LoRA for Memory Efficiency
# # quantization_config = BitsAndBytesConfig(
# #     load_in_4bit=True,
# #     bnb_4bit_quant_type="nf4",
# #     bnb_4bit_compute_dtype=torch.bfloat16,
# #     bnb_4bit_use_double_quant=True,
# # )

# # # Define LoRA Configuration
# # lora_config = LoraConfig(
# #     r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
# #     lora_alpha=16, lora_dropout=0.05, bias="none"
# # )

# # LoRA Configuration
# lora_config = LoraConfig(
#     r=16,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Apply LoRA to key projection layers
#     lora_alpha=16,
#     lora_dropout=0.05,
#     bias="none"
# )


# # Load LoRA Model AFTER Base Model
# model = get_peft_model(model, lora_config)

# Initialize WanB

In [18]:
from peft import LoraConfig, get_peft_model

# ✅ Load Model & Tokenizer
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto", low_cpu_mem_usage=True)

lora_config = LoraConfig(r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_alpha=16, lora_dropout=0.05, bias="none")
model = get_peft_model(model, lora_config)

# ✅ Configure Training
wandb.init(project="llama3_finetuning", name="llama3_ft_lora4bit", config={"batch_size": 2})
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, max_length=126, return_tensors="pt", label_pad_token_id=-100)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset, dataset_text_field="text", max_seq_length=126,
    data_collator=data_collator, dataset_num_proc=1, packing=False,
    args=TrainingArguments(per_device_train_batch_size=2, gradient_accumulation_steps=4, max_steps=100, fp16=False, bf16=True, logging_steps=10, output_dir="outputs")
)

trainer.train()

# ✅ Save Model
save_path = "/content/drive/MyDrive/EmotionCompanion/models/FINAL_Llama3_finetuned"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to {save_path}!")

Unsloth: Tokenizing ["text"]:   0%|          | 0/7000 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
10,3.025800
20,2.816800
30,2.574600
40,2.330200
50,2.292200
60,1.997300
70,2.085100
80,2.072500
90,1.959600
100,1.957800


✅ Model saved to /content/drive/MyDrive/EmotionCompanion/models/FINAL_Llama3_finetuned!


# Testing

In [36]:
# ✅ Print available columns for CBT datasets
datasets_to_check = {
    "CBT-QA": cbt_qa,
    "CBT-CD": cbt_cd,
    "CBT-PC": cbt_pc,
    "CBT-FC": cbt_fc
}

for name, dataset in datasets_to_check.items():
    print(f"\n🔹 Dataset: {name}")
    print("Columns:", dataset.column_names)


🔹 Dataset: CBT-QA
Columns: ['id', 'question', 'a', 'b', 'c', 'd', 'e']

🔹 Dataset: CBT-CD
Columns: ['id', 'ori_text', 'situation', 'thoughts', 'distortions']

🔹 Dataset: CBT-PC
Columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_major']

🔹 Dataset: CBT-FC
Columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_fine_grained']


In [38]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

# ✅ 1. Load Base LLaMA Model
base_model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # Change if needed
model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto").to("cuda")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# ✅ 2. Load Fine-Tuned LoRA Adapter
adapter_path = "/content/drive/MyDrive/EmotionCompanion/models/FINAL_Llama3_finetuned"
model = PeftModel.from_pretrained(model, adapter_path)
print("✅ LoRA adapter successfully loaded!")

# ✅ 3. Merge LoRA with Base Model
merged_model = model.merge_and_unload()

# ✅ 4. Save Full Fine-Tuned Model
save_path = "/content/drive/MyDrive/EmotionCompanion/models/Final_Fully_Finetuned_LLaMA"
merged_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Full fine-tuned model saved at: {save_path}")

# ✅ 5. Reload Merged Model
model = AutoModelForCausalLM.from_pretrained(save_path).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(save_path)

# ✅ 6. Load Multiple Datasets
print("\n✅ Loading datasets...\n")

# ✅ Load Reddit Mental Health Data
# reddit_data = load_dataset("csv", data_files="/content/drive/MyDrive/EmotionCompanion/datasets/conversations/Original Reddit Data/Labelled Data/reddit_labeled.csv", split="train[:5%]")

# ✅ Load CBT Therapy Datasets
cbt_qa = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="qa_test.json", split="train[:5%]")
cbt_cd = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="distortions_test.json", split="train[:5%]")
cbt_pc = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="core_major_test.json", split="train[:5%]")
cbt_fc = load_dataset("Psychotherapy-LLM/CBT-Bench", data_files="core_fine_test.json", split="train[:5%]")

# ✅ Load DailyDialog
daily_dialog = load_dataset("daily_dialog", split="train[:5%]", trust_remote_code=True)

# ✅ Load Empathetic Dialogues
empathetic_dialogues = load_dataset("facebook/empathetic_dialogues", split="train[:5%]", trust_remote_code=True)

print("✅ Datasets successfully loaded!")

# ✅ 7. Fix CBT Dataset Column Issue
# ✅ Extract text safely from CBT datasets
def extract_text_cbt(example, dataset_name):
    if dataset_name == "CBT-QA":
        # Use available options A, B, C, D, and E
        options = f"A) {example.get('a', '')} B) {example.get('b', '')} C) {example.get('c', '')} D) {example.get('d', '')} E) {example.get('e', '')}"
        return {"text": f"Question: {example['question']}\nOptions: {options}\nCorrect Answer: {example.get('d', 'Unknown')}"}

    elif dataset_name == "CBT-CD":
        return {"text": f"Situation: {example['situation']}\nThoughts: {example['thoughts']}\nDistortions: {example['distortions']}"}

    elif dataset_name == "CBT-PC":
        return {"text": f"Situation: {example['situation']}\nThoughts: {example['thoughts']}\nCore Belief Major: {example['core_belief_major']}"}

    elif dataset_name == "CBT-FC":
        return {"text": f"Situation: {example['situation']}\nThoughts: {example['thoughts']}\nCore Belief Fine Grained: {example['core_belief_fine_grained']}"}

    return {"text": "Unknown format"}

# ✅ Apply Fix to CBT Datasets
cbt_qa = cbt_qa.map(lambda x: extract_text_cbt(x, "CBT-QA"))
cbt_cd = cbt_cd.map(lambda x: extract_text_cbt(x, "CBT-CD"))
cbt_pc = cbt_pc.map(lambda x: extract_text_cbt(x, "CBT-PC"))
cbt_fc = cbt_fc.map(lambda x: extract_text_cbt(x, "CBT-FC"))


print("✅ Fixed CBT dataset text extraction!")

# ✅ 8. Improve Response Generation Settings
def generate_response(sample_text):
    inputs = tokenizer(sample_text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,  # More diverse responses
        top_p=0.9,        # Nucleus sampling
        repetition_penalty=1.2  # Reduce repetition
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ✅ 9. Test the Merged Model on Each Dataset
datasets_to_test = {
    # "Reddit": reddit_data,
    "CBT-QA": cbt_qa,
    "CBT-CD": cbt_cd,
    "CBT-PC": cbt_pc,
    "CBT-FC": cbt_fc,
    "DailyDialog": daily_dialog,
    "EmpatheticDialogues": empathetic_dialogues
}

for dataset_name, dataset in datasets_to_test.items():
    # Extract sample text from dataset
    sample_text = None
    if "text" in dataset.column_names:
        sample_text = dataset[0]["text"]
    elif "selftext" in dataset.column_names:  # Reddit case
        sample_text = dataset[0]["selftext"]
    elif "utterance" in dataset.column_names:  # EmpatheticDialogues case
        sample_text = dataset[0]["utterance"]
    elif "dialog" in dataset.column_names:  # DailyDialog case
        sample_text = " ".join(dataset[0]["dialog"])

    if sample_text:
        # Generate response
        response = generate_response(sample_text)
        print(f"\n🔹 **Dataset:** {dataset_name}")
        print(f"📜 **Sample Input:** {sample_text[:500]}")  # Limit text to 500 chars
        print(f"🤖 **Model Response:** {response}\n" + "-"*80)
    else:
        print(f"\n⚠️ **Skipping {dataset_name} (No valid text column found!)**")

print("\n✅ Model testing completed!")


✅ LoRA adapter successfully loaded!


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
`low_cpu_mem_usage` was None, now default to True since model is quantized.


✅ Full fine-tuned model saved at: /content/drive/MyDrive/EmotionCompanion/models/Final_Fully_Finetuned_LLaMA

✅ Loading datasets...

✅ Datasets successfully loaded!


Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

✅ Fixed CBT dataset text extraction!

🔹 **Dataset:** CBT-QA
📜 **Sample Input:** Question: Which core principle underlies Cognitive Behavioral Therapy?
Options: A) Acceptance and Commitment B) Mindfulness C) The influence of desires D) The interconnection between thoughts, feelings, and behaviors E) 
Correct Answer: The interconnection between thoughts, feelings, and behaviors
🤖 **Model Response:** Question: Which core principle underlies Cognitive Behavioral Therapy?
Options: A) Acceptance and Commitment B) Mindfulness C) The influence of desires D) The interconnection between thoughts, feelings, and behaviors E) 
Correct Answer: The interconnection between thoughts, feelings, and behaviors
Explanation: This question requires inductive reasoning as it asks the test-taker to identify a core principle underlying a specific therapeutic approach. To answer correctly, one must have knowledge of cognitive behavioral therapy (CBT) and its fundamental assumptions about human behavior. The corr

In [39]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ✅ Path to your fine-tuned LLaMA model
model_path = "/content/drive/MyDrive/EmotionCompanion/models/Final_Fully_Finetuned_LLaMA"

# ✅ Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# ✅ Test the model with an input prompt
prompt = "Can you explain what cognitive distortions are?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
outputs = model.generate(**inputs, max_new_tokens=100)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n💬 Model Response:", response)


💬 Model Response: Can you explain what cognitive distortions are? Cognitive distortions, also known as cognitive biases, are systematic errors in thinking that affect the way we process information and make decisions. They are often subtle, automatic, and habitual, leading us to misinterpret or misunderstand information.

Here are some common examples of cognitive distortions:

1. **All-or-nothing thinking**: Viewing things in absolute terms, as either all good or all bad.
2. **Black-and-white thinking**: Seeing the world in only two extremes, with no shades of gray in


In [40]:
ls -lh /content/drive/MyDrive/EmotionCompanion/models/Final_Fully_Finetuned_LLaMA

total 2.2G
-rw------- 1 root root 1.5K Mar 20 10:51 config.json
-rw------- 1 root root  234 Mar 20 10:51 generation_config.json
-rw------- 1 root root 2.1G Mar 20 10:51 model.safetensors
-rw------- 1 root root  454 Mar 20 10:51 special_tokens_map.json
-rw------- 1 root root  54K Mar 20 10:51 tokenizer_config.json
-rw------- 1 root root  17M Mar 20 10:51 tokenizer.json


In [41]:
prompt = "I'm feeling really anxious today. What should I do?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

I'm feeling really anxious today. What should I do? 
First, take a few deep breaths. This can help calm your nervous system. Try inhaling for a count of four, holding your breath for a count of four, and exhaling for a count of four. Repeat this a few times.
Next, try to identify what's causing your anxiety. Is it something specific that's happening in your life, or is it more of a general feeling? Once you know what's causing your anxiety, you can start to think about ways to address


#Evaluation

# Check Dataset Contributions (Size and Columns)

In [19]:
# ✅ Check individual dataset sizes
datasets_info = {
    "CBT_QA": len(cbt_qa),
    "CBT_CD": len(cbt_cd),
    "CBT_PC": len(cbt_pc),
    "CBT_FC": len(cbt_fc),
    "FineTome": len(fine_tome),
    "DailyDialog": len(daily_dialog),
    "EmpatheticDialogues": len(empathetic_dialogues),
}

print("✅ Dataset Sizes:")
for name, size in datasets_info.items():
    print(f"🔹 {name}: {size} samples")

# ✅ Verify dataset columns before merging
for ds, name in zip([cbt_qa, cbt_cd, cbt_pc, cbt_fc, fine_tome, daily_dialog, empathetic_dialogues],
                    ["CBT_QA", "CBT_CD", "CBT_PC", "CBT_FC", "FineTome", "DailyDialog", "EmpatheticDialogues"]):
    print(f"✅ {name} columns: {ds.column_names}")

✅ Dataset Sizes:
🔹 CBT_QA: 220 samples
🔹 CBT_CD: 146 samples
🔹 CBT_PC: 184 samples
🔹 CBT_FC: 112 samples
🔹 FineTome: 100000 samples
🔹 DailyDialog: 11118 samples
🔹 EmpatheticDialogues: 76673 samples
✅ CBT_QA columns: ['id', 'question', 'a', 'b', 'c', 'd', 'e', 'text']
✅ CBT_CD columns: ['id', 'ori_text', 'situation', 'thoughts', 'distortions', 'text']
✅ CBT_PC columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_major', 'text']
✅ CBT_FC columns: ['id', 'ori_text', 'situation', 'thoughts', 'core_belief_fine_grained', 'text']
✅ FineTome columns: ['conversations', 'source', 'score', 'text']
✅ DailyDialog columns: ['text']
✅ EmpatheticDialogues columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags', 'text']


#Check Dataset Contributions (Size and Columns)

In [20]:
# ✅ Check if dataset contains expected text samples
import random
print("✅ Sample from Merged Training Dataset:\n")
for i in range(3):
    sample_text = merged_dataset[random.randint(0, len(merged_dataset) - 1)]["text"]
    print(f"🔹 Sample {i+1}:\n{sample_text[:500]}\n{'-'*80}")

✅ Sample from Merged Training Dataset:

🔹 Sample 1:
<|start_header_id|>user<|end_header_id|>

human: Write Python code to solve the task:
You are given string s of length n consisting of 0-s and 1-s. You build an infinite string t as a concatenation of an infinite number of strings s, or t = ssss ... For example, if s = 10010, then t = 100101001010010...

Calculate the number of prefixes of t with balance equal to x. The balance of some string q is equal to cnt_{0, q} - cnt_{1, q}, where cnt_{0, q} is the number of occurrences of 0 in q, and cnt_
--------------------------------------------------------------------------------
🔹 Sample 2:
<|start_header_id|>user<|end_header_id|>

human: Using the axiomatic definition of area, how can we establish the formula for the area of a rectangle, i.e., Area = bh/2, where b is the base and h is the altitude?
gpt: To establish the formula for the area of a rectangle using the axiomatic definition of area, we can proceed as follows:

1. Consider a g

#Run Inference on Test Prompts

In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ✅ Load fine-tuned model
model_path = "/content/drive/MyDrive/EmotionCompanion/models/FINAL_Llama3_finetuned"
model = AutoModelForCausalLM.from_pretrained(model_path).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# ✅ Define test inputs
test_prompts = [
    "I'm feeling really down today, and I don't know what to do.",
    "Can you explain what cognitive distortions are?",
    "How does therapy help someone with depression?",
    "Tell me something to help with anxiety."
]

# ✅ Generate model responses
for i, prompt in enumerate(test_prompts):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\n🔹 **Test {i+1}:** {prompt}")
    print(f"💬 **Model Response:** {response}\n{'-'*80}")

`low_cpu_mem_usage` was None, now default to True since model is quantized.



🔹 **Test 1:** I'm feeling really down today, and I don't know what to do.
💬 **Model Response:** I'm feeling really down today, and I don't know what to do. I'm feeling overwhelmed by my emotions, and I don't know how to process them. I feel like I'm stuck in a rut and I don't know how to get out of it. I'm feeling really anxious and stressed, and I don't know how to calm myself down. I'm feeling like I'm losing control and I don't know how to regain it. I just wish I could talk to someone about this and get some help.

It's okay to feel this way. It
--------------------------------------------------------------------------------

🔹 **Test 2:** Can you explain what cognitive distortions are?
💬 **Model Response:** Can you explain what cognitive distortions are? 
Cognitive distortions are negative and unhelpful thought patterns that can lead to irrational beliefs, behaviors, and attitudes. They are often the result of cognitive biases, emotional states, or learned behaviors. Here are som

# Compute BLEU Score

In [23]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ✅ Sample test set (Real input + Reference output)
test_samples = [
    {
        "input": "I'm feeling really anxious today. What should I do?",
        "reference": "Try deep breathing exercises and mindfulness to calm your mind.",
    },
    {
        "input": "What are cognitive distortions?",
        "reference": "Cognitive distortions are irrational thoughts that influence emotions.",
    },
]

# ✅ Compute BLEU Scores
def compute_bleu(reference, generated):
    reference_tokens = [reference.split()]
    generated_tokens = generated.split()
    return sentence_bleu(reference_tokens, generated_tokens, smoothing_function=SmoothingFunction().method1)

# ✅ Generate Model Responses & Evaluate BLEU
bleu_scores = []
for sample in test_samples:
    inputs = tokenizer(sample["input"], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=50)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    bleu = compute_bleu(sample["reference"], generated_text)
    bleu_scores.append(bleu)

    print(f"\n🔹 **Input:** {sample['input']}")
    print(f"💬 **Generated:** {generated_text}")
    print(f"✅ **BLEU Score:** {bleu:.4f}")
    print("-" * 80)

# ✅ Average BLEU Score
print(f"\n🏆 **Final BLEU Score:** {sum(bleu_scores) / len(bleu_scores):.4f}")


🔹 **Input:** I'm feeling really anxious today. What should I do?
💬 **Generated:** I'm feeling really anxious today. What should I do? 

First, take some deep breaths and try to relax. Sometimes, our bodies can get a bit mixed up and start feeling anxious, but with some simple techniques, you can calm yourself down.

Here are some tips that might help:

1.
✅ **BLEU Score:** 0.0053
--------------------------------------------------------------------------------

🔹 **Input:** What are cognitive distortions?
💬 **Generated:** What are cognitive distortions? Cognitive distortions are negative thought patterns or ways of thinking that can cause problems for individuals in their daily lives. They can lead to mental health issues, such as anxiety, depression, and low self-esteem, as well as difficulties in relationships, work,
✅ **BLEU Score:** 0.0217
--------------------------------------------------------------------------------

🏆 **Final BLEU Score:** 0.0135


# Compute ROUGE Score

In [29]:
# ✅ Install the correct package
!pip install rouge_score --quiet

# ✅ Import the new method
from evaluate import load

# ✅ Load ROUGE metric
rouge = load("rouge")

# ✅ Define test samples
test_samples = [
    {"input": "What is cognitive therapy?", "reference": "Cognitive therapy is a type of psychotherapy that focuses on changing negative thought patterns."},
    {"input": "How do I manage anxiety?", "reference": "You can manage anxiety by practicing mindfulness, breathing exercises, and seeking professional help."},
]

# ✅ Compute ROUGE Scores
for sample in test_samples:
    inputs = tokenizer(sample["input"], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    scores = rouge.compute(predictions=[generated_text], references=[sample["reference"]])

    print(f"\n🔹 **Input:** {sample['input']}")
    print(f"💬 **Generated:** {generated_text}")
    print(f"📖 **ROUGE Score:** {scores}")
    print("-" * 80)

  Preparing metadata (setup.py) ... done

🔹 **Input:** What is cognitive therapy?
💬 **Generated:** What is cognitive therapy? Cognitive therapy is a type of psychotherapy that focuses on helping individuals understand and change negative thought patterns and behaviors that contribute to their mental health issues. It is based on the idea that our thoughts, feelings, and behaviors are interconnected, and that by changing our thoughts, we can improve our feelings and behaviors.
Cognitive therapy is often used to treat a range of mental health conditions, including depression, anxiety, post-traumatic stress disorder (PTSD), and obsessive-compulsive disorder (OCD). The
📖 **ROUGE Score:** {'rouge1': np.float64(0.2828282828282828), 'rouge2': np.float64(0.22680412371134023), 'rougeL': np.float64(0.26262626262626265), 'rougeLsum': np.float64(0.26262626262626265)}
--------------------------------------------------------------------------------

🔹 **Input:** How do I manage anxiety?
💬 **Generate

# Compute Perplexity (PPL)

In [30]:
import math
import torch

def compute_perplexity(text):
    encodings = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        loss = model(**encodings, labels=encodings["input_ids"]).loss
    return math.exp(loss.item())

# ✅ Test Perplexity on sample sentences
for sample in test_samples:
    ppl = compute_perplexity(sample["input"])
    print(f"🔹 **Perplexity Score for '{sample['input']}':** {ppl:.2f}")

# ✅ Lower Perplexity is better

🔹 **Perplexity Score for 'What is cognitive therapy?':** 135.75
🔹 **Perplexity Score for 'How do I manage anxiety?':** 77.33


# Compute F1-Score for Classification Tasks

In [31]:
from sklearn.metrics import f1_score

true_labels = []
predicted_labels = []

# ✅ Generate Model Predictions
for sample in valid_dataset.select(range(100)):  # Using 100 samples
    inputs = tokenizer(sample["text"], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=50)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract ground truth label
    if "label" in sample:
        true_labels.append(sample["label"])
    else:
        true_labels.append("Unknown")

    predicted_labels.append(generated_text)

# ✅ Compute F1 Score
f1 = f1_score(true_labels, predicted_labels, average="weighted")
print(f"\n✅ **Model F1 Score:** {f1:.4f}")


✅ **Model F1 Score:** 0.0000


# previous


In [ ]:
# import torch
# import gc
# from unsloth.chat_templates import train_on_responses_only

# # ✅ Step 1: Free up GPU memory before training
# gc.collect()
# torch.cuda.empty_cache()

# # ✅ Step 2: Function to dynamically adjust batch size based on available VRAM
# def get_safe_batch_size():
#     free_mem = torch.cuda.mem_get_info()[0] / 1e9  # Convert bytes to GB
#     if free_mem > 20:
#         return 4  # High memory → batch size 4
#     elif free_mem > 10:
#         return 2  # Medium memory → batch size 2
#     else:
#         return 1  # Low memory → batch size 1 (prevents crashes)

# # ✅ Step 3: Set up Data Collator
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer,
#     padding=True,
#     max_length=512,  # 🔹 Reduce slightly for better memory efficiency
#     return_tensors="pt",
#     label_pad_token_id=-100,
# )

# # ✅ Step 4: Apply LoRA to enable fine-tuning on quantized model
# model = FastLanguageModel.get_peft_model(
#     model,
#     r=16,  # LoRA rank
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
#                     "gate_proj", "up_proj", "down_proj"],
#     lora_alpha=16,
#     lora_dropout=0,
#     bias="none",
#     use_gradient_checkpointing="unsloth",
#     random_state=3407,
#     use_rslora=False,
#     loftq_config=None,
# )

# # ✅ Step 5: Configure Trainer
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     dataset_text_field="text",
#     max_seq_length=512,  # 🔹 Keep sequence length reasonable
#     data_collator=data_collator,
#     dataset_num_proc=1,  # 🔹 Reduce to 1 to avoid excessive memory usage
#     packing=False,

#     args=TrainingArguments(
#         per_device_train_batch_size=get_safe_batch_size(),  # 🔹 Use dynamic batch size
#         gradient_accumulation_steps=4,
#         warmup_steps=5,
#         max_steps=500,  # 🔹 Reduce from 1000 to 500 to prevent OOM
#         learning_rate=2e-4,
#         fp16=True,
#         logging_steps=10,
#         optim="adamw_8bit",
#         weight_decay=0.01,
#         lr_scheduler_type="linear",
#         seed=3407,
#         output_dir="outputs",
#         report_to="none",
#     ),
# )

# # ✅ Step 6: Apply LoRA Training Optimizations
# trainer = train_on_responses_only(
#     trainer,
#     instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
#     response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
# )

# # ✅ Step 7: Set `use_cache=False` to prevent memory issues
# model.config.use_cache = False

# # ✅ Step 8: Start Training
# trainer.train()

In [17]:
# # Initialize Trainer
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=valid_dataset,
#     dataset_text_field="text",
#     max_seq_length=256,
#     args=training_args,
#     compute_metrics=lambda eval_pred: {"eval_loss": eval_pred.loss}  # ✅ Force eval_loss logging

# )

# # ✅ Check for the latest checkpoint
# last_checkpoint = None
# if os.path.exists(checkpoint_dir):
#     checkpoints = sorted([ckpt for ckpt in os.listdir(checkpoint_dir) if "checkpoint" in ckpt])
#     if checkpoints:
#         last_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])  # Load the latest checkpoint
#         print(f"✅ Resuming training from {last_checkpoint}")
#     else:
#         print("🚀 No previous checkpoint found. Starting fresh training.")

# # ✅ Fix: Free up GPU memory before training
# torch.cuda.empty_cache()

# # ✅ Fix: Ensure `use_cache=False` to prevent memory issues
# model.config.use_cache = False

# # ✅ Fix: Ensure Model Parameters Require Gradients
# for param in model.parameters():
#     if param.dtype in [torch.float32, torch.float16, torch.bfloat16]:
#         param.requires_grad = True

# # ✅ Fix: Force optimizer to reset if needed
# if last_checkpoint:
#     print("⚠️ Resetting optimizer state to avoid mismatch issues!")
#     trainer.args.optim = "adamw_8bit"  # Ensure optimizer is consistent

# # ✅ Start training (ignoring optimizer mismatches)
# trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)

# # ✅ Step 8: Save the final fine-tuned model separately
# final_model_path = "/content/drive/MyDrive/EmotionCompanion/models/LLAMA_FINAL_FINETUNED"
# if not os.path.exists(final_model_path):
#     os.makedirs(final_model_path)

# trainer.save_model(final_model_path)
# tokenizer.save_pretrained(final_model_path)

# print(f"✅Fine-tuned model saved to {final_model_path}!")

In [1]:
# import datetime

# # Generate a unique timestamped directory name for saving checkpoints & models
# timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# save_dir = f"/content/drive/MyDrive/EmotionCompanion/models/LLaMA_FT_{timestamp}"
# checkpoint_dir = f"/content/drive/MyDrive/EmotionCompanion/checkpoints/LLaMA_FT_{timestamp}"

# # Create directories if they don’t exist
# os.makedirs(save_dir, exist_ok=True)
# os.makedirs(checkpoint_dir, exist_ok=True)

# # Update Training Arguments to store checkpoints in the new directory
# training_args.output_dir = checkpoint_dir

# # Initialize Trainer
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=valid_dataset,
#     dataset_text_field="text",
#     max_seq_length=128,
#     args=training_args,
#     compute_metrics=lambda eval_pred: {"eval_loss": eval_pred.loss}  # ✅ Force eval_loss logging
# )

# # Check for latest checkpoint inside the new checkpoint directory
# last_checkpoint = None
# if os.path.exists(checkpoint_dir):
#     checkpoints = sorted([ckpt for ckpt in os.listdir(checkpoint_dir) if "checkpoint" in ckpt])
#     if checkpoints:
#         last_checkpoint = os.path.join(checkpoint_dir, checkpoints[-1])  # Load latest checkpoint
#         print(f"Resuming training from {last_checkpoint}")
#     else:
#         print("No previous checkpoint found. Starting fresh training.")

# # Free up GPU memory before training
# torch.cuda.empty_cache()
# model.config.use_cache = False  # Prevents memory issues

# # Ensure Model Parameters Require Gradients
# for param in model.parameters():
#     if param.dtype in [torch.float32, torch.float16, torch.bfloat16]:
#         param.requires_grad = True

# # Force optimizer reset if needed
# if last_checkpoint:
#     print("⚠️ Resetting optimizer state to avoid mismatch issues!")
#     trainer.args.optim = "adamw_8bit"

# # Start training (resume from latest checkpoint if available)
# trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)

# # Step 8: Save the final fine-tuned model to a new folder
# final_model_path = os.path.join(save_dir, "FINAL_MODEL")
# os.makedirs(final_model_path, exist_ok=True)

# trainer.save_model(final_model_path)
# tokenizer.save_pretrained(final_model_path)

# print(f"Fine-tuned model saved to {final_model_path}!")

In [ ]:
import os
import shutil

# Define backup path
latest_backup_path = "/content/drive/MyDrive/EmotionCompanion/models/LATEST_CHECKPOINT"

# ✅ Backup latest checkpoint before saving final model
if os.path.exists(checkpoint_dir):
    shutil.copytree(checkpoint_dir, latest_backup_path, dirs_exist_ok=True)
    print(f"✅ Latest checkpoint copied to: {latest_backup_path}")
else:
    print("❌ No checkpoint found to copy.")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# ✅ Load the fine-tuned model
final_model_path = os.path.join(save_dir, "FINAL_MODEL")
model = AutoModelForCausalLM.from_pretrained(final_model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(final_model_path)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

# ✅ Test inference with the fine-tuned model
prompt = "I am feeling very anxious about my future. What should I do?"
result = pipe(prompt, max_length=200, do_sample=True)

print("\n🔹 Model Response:")
print(result[0]["generated_text"])

# ✅ Finish WandB logging
wandb.finish()

In [ ]:
import os

# ✅ Save the fine-tuned model with a timestamped directory
final_save_path = os.path.join(save_dir, "Llama3_Finetuned")
os.makedirs(final_save_path, exist_ok=True)

model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f"✅ Final fine-tuned model saved to {final_save_path}!")

In [ ]:
# # ✅ Step 9: Backup latest checkpoint (Optional)
# backup_path = "/content/drive/MyDrive/EmotionCompanion/models/LATEST_CHECKPOINT"
# if os.path.exists(checkpoint_dir):
#     shutil.copytree(checkpoint_dir, backup_path, dirs_exist_ok=True)
#     print(f"✅ Checkpoint copied to: {backup_path}")
# else:
#     print("❌ No checkpoint found to copy.")

# # ✅ Step 10: Load & Test the fine-tuned model
# model = AutoModelForCausalLM.from_pretrained(final_model_path, torch_dtype=torch.float16, device_map="auto")
# pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

# prompt = "I am feeling very anxious about my future. What should I do?"
# result = pipe(prompt, max_length=200, do_sample=True)

# print("\n🔹 Model Response:")
# print(result[0]["generated_text"])

# # ✅ Step 11: Finish WandB Logging
# wandb.finish()

✅ Checkpoint copied to: /content/drive/MyDrive/EmotionCompanion/models/LATEST_CHECKPOINT


In [ ]:
# # save_path = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned"
# save_path = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned_again"

# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)

# print(f"Model saved to {save_path}!")

# Start API

In [ ]:
!ngrok authtoken 2uCaFGsZsUvpzNzGSz2FwnY89L5_36cUAfeXMHVXK7WMqzkRm

/bin/bash: line 1: ngrok: command not found


In [ ]:
# Install dependencies
!pip install fastapi uvicorn pyngrok nest_asyncio -q

# Import libraries
from fastapi import FastAPI
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import os

# Authenticate ngrok
NGROK_AUTH_TOKEN = "2uCaFGsZsUvpzNzGSz2FwnY89L5_36cUAfeXMHVXK7WMqzkRm"

# Load fine-tuned LLaMA model
MODEL_PATH = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16, device_map="auto")
model.eval()

# Start FastAPI Server
app = FastAPI()

@app.post("/generate")
def generate_response(data: dict):
    """Accepts text input and generates an emotion-aware response using LLaMA."""
    text = data["text"]
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=1.0)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    response = generated_text.split("assistant\n\n")[-1]  # Extract assistant's response
    return {"response": response}

# Start ngrok with authentication
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000).public_url
print(f"🚀 LLaMA API running at: {public_url}")

# Patch asyncio to prevent the RuntimeError
nest_asyncio.apply()

# Run the API in Google Colab using Uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.4 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

INFO:     Started server process [1866]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 LLaMA API running at: https://c975-34-124-136-239.ngrok-free.app
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     116.15.190.249:0 - "POST /generate HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1866]


# Train LlaMA Model (Current)

In [ ]:
# from transformers import TrainingArguments, DataCollatorForSeq2Seq
# from trl import SFTTrainer

# # Load Model & Tokenizer
# model_path = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned"
# tokenizer = AutoTokenizer.from_pretrained(model_path)
# model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# print("Model and tokenizer loaded successfully!")

# # Set up Data Collator
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer,
#     padding=True,
#     max_length=512,
#     return_tensors="pt",
#     label_pad_token_id=-100,
# )

# # Define Training Arguments
# training_args = TrainingArguments(
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     warmup_steps=5,
#     max_steps=1000,  # Increase for better results
#     learning_rate=2e-4,
#     fp16=True,
#     logging_steps=10,
#     optim="adamw_8bit",
#     weight_decay=0.01,
#     lr_scheduler_type="linear",
#     seed=3407,
#     output_dir="outputs",
#     report_to="none",
# )

# # Initialize Trainer
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=merged_dataset,
#     dataset_text_field="text",
#     max_seq_length=512,
#     data_collator=data_collator,
#     dataset_num_proc=2,
#     packing=False,
#     args=training_args,
# )

# # Apply `train_on_responses_only` after defining `trainer`
# trainer = train_on_responses_only(
#     trainer,
#     instruction_part="<|start_header_id|>user<|end_header_id|>\n",
#     response_part="<|start_header_id|>assistant<|end_header_id|>\n",
# )

# Evaluation Metrics

In [ ]:
# ## Model Evaluation (Classfication & BLEU Score)

# # Load Sentence Transformer for Semantic Similarity
# similarity_model = SentenceTransformer("all-MiniLM-L6-v2")

# # Collect Predictions & References
# predictions, references = [], []
# for example in valid_dataset:
#     inputs = tokenizer(example["text"], return_tensors="pt").to("cuda")
#     outputs = model.generate(**inputs, max_new_tokens=64)

#     generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     predictions.append(generated_text)
#     references.append(example["text"])

# # Compute Accuracy & F1-Score
# accuracy = accuracy_score(references, predictions)
# f1 = f1_score(references, predictions, average="weighted", zero_division=0)

# print("\nClassification Report:")
# print(classification_report(references, predictions, zero_division=0))

# print(f"\nAccuracy: {accuracy:.4f}")
# print(f"F1 Score: {f1:.4f}")
# print("\nConfusion Matrix:")
# print(confusion_matrix(references, predictions))

# # Compute BLEU Score
# bleu_scores = []
# for true, pred in zip(references, predictions):
#     reference = [true.split()]
#     candidate = pred.split()
#     bleu = sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1)
#     bleu_scores.append(bleu)

# avg_bleu = np.mean(bleu_scores)

# # Compute Semantic Similarity
# true_embeddings = similarity_model.encode(references)
# pred_embeddings = similarity_model.encode(predictions)
# semantic_scores = [util.cos_sim(true, pred).item() for true, pred in zip(true_embeddings, pred_embeddings)]
# avg_semantic_similarity = np.mean(semantic_scores)

# print(f"\nBLEU Score: {avg_bleu:.4f}")
# print(f"Semantic Similarity: {avg_semantic_similarity:.4f}")

# Run Inference on Emotion Aware Chatbot

In [ ]:
save_path = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}!")

In [ ]:
# # # Run Inference
# # FastLanguageModel.for_inference(model)  # Enable optimized inference

# # messages = [
# #     {"role": "user", "content": "I'm feeling really stressed today. What should I do?"},
# # ]

# # inputs = tokenizer.apply_chat_template(
# #     messages,
# #     tokenize=True,
# #     add_generation_prompt=True,
# #     return_tensors="pt",
# # ).to("cuda")

# # # Generate response
# # outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=1.5)

# # # Remove system prompts & format properly
# # generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# # generated_text = generated_text.split("assistant\n\n")[-1]  # Extract assistant's response only

# # print("**Model Response:**", generated_text)

# # messages = [
# #     {"role": "user", "content": "I'm feeling really stressed today. What should I do?"},
# # ]

# # inputs = tokenizer.apply_chat_template(
# #     messages,
# #     tokenize=True,
# #     add_generation_prompt=True,
# #     return_tensors="pt",
# # ).to("cuda")

# # # Generate response
# # outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=1.0)

# # # Extract assistant response
# # generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# # generated_text = generated_text.split("assistant\n\n")[-1]  # Extract assistant's response

# # print("**Model Response:**", generated_text)

# messages = [
#     {"role": "user", "content": "I'm feeling really stressed today. What should I do?"},
# ]

# inputs = tokenizer.apply_chat_template(
#     messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
# ).to("cuda")

# # Generate response
# outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True, temperature=1.0)

# # Extract assistant response
# generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# generated_text = generated_text.split("assistant\n\n")[-1]  # Extract assistant's response

# print("**Model Response:**", generated_text)

# Save & Export Model

# Model Evaluation

In [ ]:
# ## Model Evaluation (Classfication & BLEU Score)

from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Load Fine-Tuned Model
model_path = "/content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Fine-tuned model loaded successfully!")

# Load Semantic Similarity Model
similarity_model = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L6-v2")

# Batch Processing
batch_size = 8  # Adjust based on your GPU memory
predictions, references = [], []

for i in range(0, len(valid_dataset), batch_size):
    batch_texts = valid_dataset[i:i+batch_size]["text"]

    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=32)

    generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    predictions.extend(generated_texts)
    references.extend(batch_texts)

# Compute Accuracy & F1-Score
accuracy = accuracy_score(references, predictions)
f1 = f1_score(references, predictions, average="weighted", zero_division=0)

print("\nClassification Report:")
print(classification_report(references, predictions, zero_division=0))
print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(references, predictions))

# Compute BLEU Score (Batch Optimized)
bleu_scores = [
    sentence_bleu([ref.split()], pred.split(), smoothing_function=SmoothingFunction().method1)
    for ref, pred in zip(references, predictions)
]
avg_bleu = np.mean(bleu_scores)

# Compute Semantic Similarity (Batch Optimized)
true_embeddings = similarity_model.encode(references, batch_size=16, convert_to_tensor=True)
pred_embeddings = similarity_model.encode(predictions, batch_size=16, convert_to_tensor=True)

semantic_scores = util.cos_sim(true_embeddings, pred_embeddings).cpu().numpy().diagonal()
avg_semantic_similarity = np.mean(semantic_scores)

print(f"\nBLEU Score: {avg_bleu:.4f}")
print(f"Semantic Similarity: {avg_semantic_similarity:.4f}")

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-16' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/main.py", line 579, in run
    server.run()
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run
    s

Fine-tuned model loaded successfully!


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

OutOfMemoryError: CUDA out of memory. Tried to allocate 688.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 626.12 MiB is free. Process 77578 has 14.13 GiB memory in use. Of the allocated memory 13.80 GiB is allocated by PyTorch, and 192.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!zip -r Llama3_finetuned.zip /content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned

  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/ (stored 0%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/README.md (deflated 66%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/generation_config.json (deflated 38%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/tokenizer_config.json (deflated 94%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/adapter_config.json (deflated 54%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/special_tokens_map.json (deflated 71%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/tokenizer.json (deflated 85%)
  adding: content/drive/MyDrive/EmotionCompanion/models/Llama3_finetuned/adapter_model.safetensors (deflated 45%)


In [ ]:
from google.colab import files
files.download("Llama3_finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Checking DataSets

In [ ]:
# print("Dataset Sizes Before Merging:")
# print("Raw Reddit Data:", len(raw_reddit_data))
# print("Labeled Reddit Data:", len(labeled_reddit_data))
# print("CBT QA:", len(cbt_qa))
# print("CBT CD:", len(cbt_cd))
# print("CBT PC:", len(cbt_pc))
# print("CBT FC:", len(cbt_fc))
# print("FineTome:", len(fine_tome))
# print("DailyDialog:", len(daily_dialog))
# print("Empathetic Dialogues:", len(empathetic_dialogues))

# print("\nDataset Sizes After Merging:")
# print("Merged Dataset Count:", len(merged_dataset))

In [ ]:
# from datasets import Dataset, concatenate_datasets

# # 🔹 Step 1: Convert Pandas DataFrames to Hugging Face Datasets (only if needed)
# def convert_to_hf_dataset(data):
#     """Convert Pandas DataFrame to Hugging Face Dataset, if not already in Dataset format."""
#     if isinstance(data, pd.DataFrame):  # Only convert if it's a Pandas DataFrame
#         return Dataset.from_pandas(data)
#     return data  # Return as-is if it's already a Dataset

# # Convert datasets only if they are Pandas DataFrames
# raw_reddit_data = convert_to_hf_dataset(raw_reddit_data)
# labeled_reddit_data = convert_to_hf_dataset(labeled_reddit_data)
# cbt_qa = convert_to_hf_dataset(cbt_qa)
# cbt_cd = convert_to_hf_dataset(cbt_cd)
# cbt_pc = convert_to_hf_dataset(cbt_pc)
# cbt_fc = convert_to_hf_dataset(cbt_fc)
# fine_tome = convert_to_hf_dataset(fine_tome)
# daily_dialog = convert_to_hf_dataset(daily_dialog)
# empathetic_dialogues = convert_to_hf_dataset(empathetic_dialogues)

# # 🔹 Step 2: Ensure All Datasets Have a "text" Column
# def ensure_text_column(dataset):
#     """Ensures each dataset has a 'text' column with correctly formatted strings."""
#     if "text" in dataset.column_names:
#         return dataset  # Already correct format

#     elif "conversations" in dataset.column_names:
#         dataset = dataset.map(lambda x: {"text": " ".join([turn["value"] for turn in x["conversations"] if "value" in turn])})
#         dataset = dataset.remove_columns(["conversations"])

#     elif "title" in dataset.column_names and "selftext" in dataset.column_names:
#         dataset = dataset.map(lambda x: {"text": f"user:\n{x['title']}\n{x['selftext']}\n\nassistant:\n"})
#         dataset = dataset.remove_columns(["title", "selftext"])

#     elif "question" in dataset.column_names:
#         dataset = dataset.map(lambda x: {
#             "text": f"user:\n{x['question']}\nA) {x.get('a', '')} B) {x.get('b', '')} C) {x.get('c', '')} D) {x.get('d', '')}\n\nassistant:\n{x.get('d', 'Unknown Answer')}"
#         })
#         dataset = dataset.remove_columns(["question", "a", "b", "c", "d"])

#     elif "utterance" in dataset.column_names:
#         dataset = dataset.map(lambda x: {"text": x["utterance"]})
#         dataset = dataset.remove_columns(["utterance"])

#     else:
#         dataset = dataset.map(lambda x: {"text": ""})

#     return dataset

# # Apply text column standardization
# raw_reddit_data = ensure_text_column(raw_reddit_data)
# labeled_reddit_data = ensure_text_column(labeled_reddit_data)
# cbt_qa = ensure_text_column(cbt_qa)
# cbt_cd = ensure_text_column(cbt_cd)
# cbt_pc = ensure_text_column(cbt_pc)
# cbt_fc = ensure_text_column(cbt_fc)
# fine_tome = ensure_text_column(fine_tome)
# daily_dialog = ensure_text_column(daily_dialog)
# empathetic_dialogues = ensure_text_column(empathetic_dialogues)

# # 🔹 Step 3: Remove Extra Columns (Keep Only "text")
# def keep_only_text(dataset):
#     """Removes all columns except 'text' for consistent merging."""
#     return dataset.remove_columns([col for col in dataset.column_names if col != "text"])

# datasets_to_merge = [
#     keep_only_text(raw_reddit_data),
#     keep_only_text(labeled_reddit_data),
#     keep_only_text(cbt_qa),
#     keep_only_text(cbt_cd),
#     keep_only_text(cbt_pc),
#     keep_only_text(cbt_fc),
#     keep_only_text(fine_tome),
#     keep_only_text(daily_dialog),
#     keep_only_text(empathetic_dialogues)
# ]

# # 🔹 Step 4: Merge the Datasets
# print("\n🔹 Merging datasets...")
# merged_dataset = concatenate_datasets(datasets_to_merge)

# # 🔹 Step 5: Filter Empty "text" Entries
# merged_dataset = merged_dataset.filter(lambda ex: ex["text"].strip() != "")

# # 🔹 Step 6: Final Checks
# print(f"✅ Merged Dataset Count: {len(merged_dataset)}")
# print(f"✅ Expected Total Dataset Count: {sum(len(ds) for ds in datasets_to_merge)}")

# # 🔹 Step 7: Print Sample Entries
# print("\n🔹 Checking first 5 samples in merged dataset...")
# for i in range(5):
#     print(f"Sample {i}:\n{merged_dataset[i]['text']}\n{'-'*80}")